### Run Dependencies

In [ ]:
%run Legal_-_File_Formats_And_Processing

### Ensure Central Error Log Table Exists

In [ ]:
# ── Ensure the error_legal managed table exists for central error logs ─────────
#    (managed table under error_legal schema — no explicit path needed)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {Error_Schema}.{CENTRAL_ERROR_LOG_TABLE} (
        Error_ID          STRING,
        Source_Table      STRING,
        Pipeline_Layer    STRING,
        Error_Message     STRING,
        Error_Record_JSON STRING,
        Error_Logged_Time TIMESTAMP
    )
    USING DELTA
""")
print(f"Table ready: {Error_Schema}.{CENTRAL_ERROR_LOG_TABLE}")


### Main Loop — RAW → Bronze (Managed Tables)

In [ ]:
import uuid

# ── Pre-build PK dict from cached lookup (zero extra Spark jobs) ──────────────
# pk_lookup and uk_lookup are built in File_Formats_And_Processing

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP  —  RAW → BRONZE
# ══════════════════════════════════════════════════════════════════════════════
for File_Type in CSV_Files:

    Table_Name = None
    audit_id   = None

    try:
        Table_Name   = re.search(r"([^/]+)(?=\.\w+$)", File_Type).group(1)
        File_Name    = File_Type.split("/")[-1]
        extension    = File_Name.rsplit(".", 1)[-1]
        Updated_Date = datetime.now().strftime("%Y-%m-%d")
        metadata_id  = str(uuid.uuid4())
        audit_id     = str(uuid.uuid4())
        bronze_table = f"{Bronze_Schema}.{Table_Name}"

        # ── Build schema and read CSV ──────────────────────────────────────
        schema_generated = build_spark_schema(File_Name, Schema_Mapping_Bronze)

        Read_df = (
            spark.read.csv(File_Type, header=True, schema=schema_generated)
            .withColumn("Deletion_Flag",    lit("N"))
            .withColumn("Updated_Date",     lit(Updated_Date))
            .withColumn("Uploaded_Datetime", current_timestamp())
        )

        # ── Primary-key null / invalid check ──────────────────────────────
        PK = pk_lookup.get(File_Name)
        UK = uk_lookup.get(File_Name)

        if PK:
            Read_df = Read_df.withColumn(
                "Deletion_Flag",
                when(
                    col(PK).isNull()
                    | (trim(col(PK).cast("string")) == "")
                    | (upper(trim(col(PK).cast("string"))) == "NULL")
                    | (upper(trim(col(PK).cast("string"))) == "NA")
                    | (trim(col(PK).cast("string")) == "0"),
                    lit("Y"),
                ).otherwise(col("Deletion_Flag")),
            )
        else:
            print(f"[WARN] No PK defined for {File_Name}")

        # ── Duplicate / unique-key dedup ───────────────────────────────────
        if UK:
            window_spec = Window.partitionBy(UK).orderBy(UK)
            Read_df = (
                Read_df.withColumn("_row_num", row_number().over(window_spec))
                .withColumn(
                    "Deletion_Flag",
                    when(col("_row_num") > 1, lit("Y")).otherwise(col("Deletion_Flag")),
                )
                .drop("_row_num")
            )
        else:
            print(f"[WARN] No UK defined for {File_Name}")

        # ── Split valid / error rows ───────────────────────────────────────
        Error_Log_df = Read_df.filter(col("Deletion_Flag") == "Y")

        if PK and error_record_count_pre := 0:
            pass  # placeholder — counted below after .cache()

        if PK:
            Error_Log_df = Error_Log_df.withColumn(
                "Comments",
                when(trim(col(PK).cast("string")) == "0",
                     lit(f"{PK}: value is 0"))
                .when(col(PK).isNull(),
                      lit(f"{PK}: NULL"))
                .when(upper(trim(col(PK).cast("string"))).isin("NULL", "NA"),
                      lit(f"{PK}: NULL/NA string"))
                .otherwise(lit(f"{PK} or UK: duplicate")),
            )

        Read_df = Read_df.filter(col("Deletion_Flag") != "Y")

        # ── Cache DataFrames ONCE — avoids duplicate Spark scans ──────────
        Read_df.cache()
        Error_Log_df.cache()
        bronze_record_count = Read_df.count()
        error_record_count  = Error_Log_df.count()

        # ── Write managed Bronze table (no path — Lakehouse Tables/) ──────
        Read_df.write \
            .mode("overwrite") \
            .format("delta") \
            .option("mergeSchema", "true") \
            .saveAsTable(bronze_table)

        Read_df.unpersist()
        print(f"[BRONZE] Written | table={bronze_table} | records={bronze_record_count}")

        # ── Audit log (SUCCESS) ────────────────────────────────────────────
        try:
            log_audit(
                audit_id          = audit_id,
                source_type       = "FILE",
                destination       = bronze_table,
                notebook_name     = "Legal - Raw_To_Bronze",
                layer_name        = "BRONZE",
                table_name        = Table_Name,
                records_processed = bronze_record_count,
                status            = "SUCCESS",
                error_message     = "",
            )
        except Exception as _ae:
            print(f"[AUDIT WARN] {_ae}")

        # ── Metadata log (UPSERT) ──────────────────────────────────────────
        upsert_bronze_metadata(
            file_name   = File_Name,
            table_name  = Table_Name,
            file_path   = File_Type,
            extension   = extension,
            schema      = Bronze_Schema,
            metadata_id = metadata_id,
        )

        # ── Central error log ──────────────────────────────────────────────
        if error_record_count > 0:
            json_cols = [col(c) for c in Error_Log_df.columns if c != "Comments"]

            Central_Error_df = Error_Log_df.select(
                expr("uuid()").alias("Error_ID"),
                lit(Table_Name).alias("Source_Table"),
                lit("RAW_TO_BRONZE").alias("Pipeline_Layer"),
                col("Comments").alias("Error_Message"),
                to_json(struct(*json_cols)).alias("Error_Record_JSON"),
                current_timestamp().alias("Error_Logged_Time"),
            )

            # Write to managed error table
            Central_Error_df.write \
                .mode("append") \
                .format("delta") \
                .option("mergeSchema", "true") \
                .saveAsTable(f"{Error_Schema}.{CENTRAL_ERROR_LOG_TABLE}")

            # Mirror to Warehouse using executemany (fast)
            try:
                _conn   = get_warehouse_conn()
                _cursor = _conn.cursor()
                _params = [
                    (r["Error_ID"], r["Source_Table"], r["Pipeline_Layer"],
                     r["Error_Message"], r["Error_Record_JSON"], r["Error_Logged_Time"])
                    for r in Central_Error_df.collect()
                ]
                _cursor.fast_executemany = True
                _cursor.executemany(
                    f"""INSERT INTO [{WAREHOUSE_SCHEMA}].[{CENTRAL_ERROR_LOG_WH}]
                        (Error_ID,Source_Table,Pipeline_Layer,
                         Error_Message,Error_Record_JSON,Error_Logged_Time)
                        VALUES (?,?,?,?,?,?)""",
                    _params,
                )
                _conn.commit()
                _conn.close()
                print(f"[ERROR LOG] Warehouse updated | table={Table_Name} | rows={len(_params)}")
            except Exception as _we:
                print(f"[ERROR LOG WARN] Warehouse insert failed: {_we}")

        Error_Log_df.unpersist()
        print(f"[DONE] {Table_Name} | valid={bronze_record_count} | errors={error_record_count}")

    except Exception as e:
        try:
            log_audit(
                audit_id          = audit_id or str(uuid.uuid4()),
                source_type       = "FILE",
                destination       = f"{Bronze_Schema}.{Table_Name}" if Table_Name else "UNKNOWN",
                notebook_name     = "Legal - Raw_To_Bronze",
                layer_name        = "BRONZE",
                table_name        = Table_Name or "UNKNOWN",
                records_processed = 0,
                status            = "FAILED",
                error_message     = str(e),
            )
        except Exception as _ae2:
            print(f"[AUDIT WARN] {_ae2}")
        print(f"[FAILED] {Table_Name} | {e}")
        raise


### Optional Files Backup

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  OPTIONAL FILES BACKUP  —  Raw → Bronze
#  Controlled by ENABLE_FILES_BACKUP in Libraries_And_Path.
# ══════════════════════════════════════════════════════════════════════════════
if ENABLE_FILES_BACKUP:
    print("[BACKUP] Starting Bronze backup...")

    _bronze_tables = [
        (re.search(r"([^/]+)(?=\.\w+$)", f).group(1), Bronze_Schema)
        for f in CSV_Files
    ]

    backup_tables_to_files(
        tables      = _bronze_tables,
        backup_root = Bronze_Backup_Path,
        layer_label = "BRONZE_BACKUP",
    )

    _prune_old_backups(Bronze_Backup_Path, BACKUP_RETENTION_DAYS)

else:
    print("[BACKUP] Skipped — ENABLE_FILES_BACKUP is False")
